# SDA-DSC-212 — Time Series Analysis and Forecasting
## Lab 4 — LightGBM Feature-Based Forecaster

**Duration** 50 minutes · **Work in pairs** · `git checkout lab4-start`

> The one rule for this whole course: every feature, every split and every scaler
> must be computable using only data that existed at the forecast origin.

### Tasks

| min | task |
|---|---|
| 5 | Find the **two leaks** in the starter code you inherited. Write them down in `LEAKS.md`. **Do not fix them yet.** |
| 12 | Build the leakage-safe design matrix with `build_features` — ACF-chosen lags, shifted rolling windows, KSA calendar flags, Fourier terms. Verify with `assert_no_leakage`. |
| 12 | Direct multi-step targets for H = 24 with `DirectLGBMForecaster`. Split **by time**. Say out loud why direct, not recursive. |
| 8 | Forecast the test window day by day. MAE and MAPE against the seasonal-naive baseline; overlay one day. |
| 8 | Feature importance by gain and by family. Interpret the top five. Importance is associational, never causal. |
| 5 | The four-design leakage demonstration. Finish `LEAKS.md`. Commit. |

In [ ]:
import sys, warnings
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")

import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 120)
plt.rcParams.update({"figure.figsize": (11, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.25})

DATA = "../data"

In [ ]:
from tayyar.data.load import load_demand, load_calendar, join_calendar

df = join_calendar(load_demand(f"{DATA}/ksa_grid_demand.csv"),
                   load_calendar(f"{DATA}/ksa_calendar.csv"))
y = df["demand_mw"].astype(float)
print(f"{len(df):,} hourly rows   {df.index.min():%Y-%m-%d} -> {df.index.max():%Y-%m-%d}")

### Task 1 — the starter code you inherited

Run the next cell. It works, and the number it prints would get a model promoted.

Your job for the next five minutes is **not** to fix it. It is to name, precisely, the
two lines that make that number meaningless — and write them into `LEAKS.md`.

In [ ]:
# ---------------------------------------------------------------------------
# The starter code you inherited
#
# A colleague left this behind on their last day, with the note:
#   "day-ahead demand model, MAPE well under 2%, ready for the control room."
#
# It runs. It reports that number. Two lines in this cell make it a lie.
# Find both before you write any new code.
# ---------------------------------------------------------------------------
from sklearn.model_selection import train_test_split
import lightgbm as lgb

# Target: demand at hour t. Day-ahead means every feature must be known 24 hours
# before t — so the lags all start at 24. That much the colleague got right.
work = df[["demand_mw", "temp_c"]].copy()
work["hour"] = work.index.hour
work["dow"]  = work.index.dayofweek
for L in (24, 25, 48, 168):
    work[f"lag{L}"] = work["demand_mw"].shift(L)
work["roll24"]     = work["demand_mw"].rolling(24, center=True, min_periods=1).mean()
work["roll24_max"] = work["demand_mw"].rolling(24, center=True, min_periods=1).max()
work = work.dropna()

FEATS = ["temp_c", "hour", "dow", "lag24", "lag25", "lag48", "lag168",
         "roll24", "roll24_max"]
Xa, Xb, ya, yb = train_test_split(work[FEATS], work["demand_mw"],
                                  test_size=0.2, random_state=0, shuffle=True)

starter = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.06, num_leaves=63,
                            verbose=-1, random_state=1).fit(Xa, ya)
starter_mape = float(np.mean(np.abs((yb - starter.predict(Xb)) / yb)) * 100)
print(f"inherited starter — day-ahead MAPE: {starter_mape:.2f}%")

In [ ]:
# TODO (5 min) — do not change the cell above yet.
#
# Two lines in the starter make the printed MAPE meaningless. One is about *how the
# data was split*; the other is about *what one feature can see*.
#
# 1. Print the first and last timestamp of Xa (train) and Xb (test).
#    How many test rows are EARLIER than the last training row?
#
# 2. Pick any timestamp t in `work`. Compare work.loc[t, "roll24"] against
#    the mean of demand_mw over the window you think it covers. Slide the window
#    until the two agree exactly. Which hours does it actually contain — and is
#    the hour you are trying to predict one of them?
#
# Write both up in LEAKS.md: the offending line, the mechanism, one sentence each.
#
# HINT: a feature is legal only if you could compute it at the forecast origin with a
#       clock on the wall. "Centred" is a word that should stop you every time.

### Task 2 — the leakage-safe design matrix

Four families, all computable at the forecast origin:

- **lags** — the significant ACF lags from Lab 2: 1, 2, 3 (short memory), 24, 25
  (yesterday's same hour and the hour after), 48, 168, 169, 336 (the weekly rhythm).
- **rolling** — `.shift(1)` **before** `.rolling(..., center=False)`, windows of 24 and 168 hours.
- **calendar** — hour, day of week, month, day of year, and the KSA flags. The weekend here is
  **Friday (4) and Saturday (5)** in pandas `dayofweek`, where Monday = 0. A model built with the
  Western Sat/Sun assumption learns the wrong two days.
- **Fourier** — smooth daily (period 24), weekly (168) and annual (8760) seasonality in a
  handful of sine/cosine columns instead of hundreds of dummies.

In [ ]:
from tayyar.features.build import (build_features, assert_no_leakage,
                                    DEFAULT_LAGS, DEFAULT_WINDOWS)

# TODO (12 min)
#   1. feat = build_features(...) on the columns
#      ["demand_mw","temp_c","is_weekend","is_ramadan","is_eid","is_national_day"]
#   2. Xcols = every column except the target; X = feat[Xcols].astype(float)
#   3. Report the shape: how many rows, how many features?
#   4. Prove the rolling guard yourself: compare feat["demand_mw_rmean24"] against
#      yy.shift(1).rolling(24, min_periods=6, center=False).mean()
#   5. Prove the weekend flag: group is_weekend by day_of_week and check it fires
#      on 4 (Friday) and 5 (Saturday), NOT on 5 and 6.
#   6. assert_no_leakage(feat[["demand_mw"] + Xcols])
#
# It must print exactly:  assert_no_leakage: PASS (44 features checked)
#
# HINT: the long lags create leading NaNs — 336 hours of them. Do not "fix" that by
#       back-filling, which invents history. Drop the rows.

### Task 3 — direct multi-step, split by time

**Recursive** is one model that predicts one step ahead and is then fed its own prediction
back in as a lag, 24 times. It is compact and it guarantees a coherent path — but the errors
compound, and after a few steps the "lags" it is consuming are predictions, whose
distribution is not the distribution of the real lags it was trained on.

**Direct** is one model per horizon `h`, each trained on the target shifted by `-h`. Nothing
compounds, because nothing is fed back. The costs are `H` models to fit and store, and no
guarantee that the 24 horizons form a coherent day — hour 13 is not constrained to sit
sensibly next to hour 14.

At a 24-hour day-ahead horizon the compounding is exactly where recursive chains fall apart,
so **direct is the robust default here**. `DirectLGBMForecaster` is direct.

In [ ]:
from tayyar.models.lgbm_forecaster import DirectLGBMForecaster

# TODO (12 min)
#   1. CUT_TRAIN = "2023-09-30 23:00". Xtr = X[:CUT_TRAIN].dropna(); ytr = yy[Xtr.index]
#      Xte = X[CUT_TRAIN:]
#   2. Print both windows and check that no training hour is later than the test window.
#      Then print len(Xtr.index.intersection(Xte.index)) and explain the answer —
#      pandas label slicing is inclusive at BOTH ends.
#      Splitting by time is the fix for leak 1.
#   3. lgbm = DirectLGBMForecaster(horizon=24).fit(Xtr, ytr)
#   4. Confirm 24 models were fitted, and read fit() in the source: what is the
#      target for model h?
#
# Then write two sentences in LEAKS.md: recursive vs direct, and why direct at h=24.
#
# HINT: the fit takes a minute or two — 24 models. That cost IS the direct strategy;
#       recursive would be one model and a loop, and would compound its own error.

### Task 4 — forecast the test window, one day at a time

One forecast origin per day at 23:00, 60 days of November and December 2023. At each origin
the model sees one row of features and emits 24 hours. The baseline is seasonal-naive:
tomorrow is a copy of the last 24 hours. `MASE < 1` means you beat that free baseline.

In [ ]:
from tayyar.eval.metrics import all_metrics

# TODO (8 min) — one forecast origin per day at 23:00 across the test window.
#
#   test_idx = X[CUT_TRAIN:].index
#   starts   = list(range(len(test_idx) - 25, 24 * 30, -24))[:60][::-1]
#
#   for each origin:
#     pos    = X.index.get_loc(origin)
#     Xo     = X.iloc[[pos]]                     # one row of features
#     actual = yy.iloc[pos+1 : pos+25]           # the 24 hours that follow
#     fc     = [lgbm.models_[h].predict(Xo)[0] for h in 1..24]
#     bl     = seasonal-naive: a copy of the previous 24 hours
#     hist   = yy.iloc[max(0, pos-24*365) : pos]  # the in-sample MASE scale
#     all_metrics(actual, fc, hist, 24)
#
#   Average the per-origin metrics for both models and print MAE, MAPE_%, MASE, bias.
#   Then overlay actual / model / baseline for one day.
#
# HINT: skip any origin where Xo has a NaN — that is a real forecast failure, not a
#       row to quietly fill. And never let `actual` touch the feature row.

### Task 5 — what the model leaned on

In [ ]:
from tayyar.models.explain import importance_by_family

# TODO (8 min)
#   1. imp = lgbm.feature_importance(25)   -- this is mean GAIN, not split count.
#      Why gain? Split count rewards high-cardinality columns for being splittable.
#   2. print the top 10, then importance_by_family(imp)
#   3. horizontal bar chart of the top 14
#   4. Interpret the top FIVE out loud to your pair — one sentence each, in domain terms.
#
# HINT: the sentence that must be said before anyone acts on this table is
#       "these are associations, not causes". Ask yourselves what happens to
#       demand_mw_lag3's gain if you delete demand_mw_rmean24.

### The leakage demonstration — one model, four evaluation designs

Same LightGBM, same data, same day-ahead target. Only the *evaluation design* changes, plus
one feature. `tools/leak_demo.py` produces these; the recorded run is in `leak_results.json`.

In [ ]:
import json

leak = json.load(open("../leak_results.json"))
designs = [
    ("A", "shuffle split + leaky features", leak["A_shuffle_plus_leaky_features"]),
    ("B", "shuffle split, clean features",  leak["B_shuffle_clean_features"]),
    ("C", "time-ordered split, clean",      leak["C_time_split_clean"]),
    ("D", "rolling-origin backtest, clean", leak["D_rolling_origin_clean"]),
]
for k, name, v in designs:
    print(f"{k}  {name:<34s} {v:5.2f}% MAPE")
print(f"\nD spread across origins: {leak['D_spread'][0]:.2f}% .. {leak['D_spread'][1]:.2f}%")
print(f"the leaked design reports a model "
      f"{leak['D_rolling_origin_clean'] / leak['A_shuffle_plus_leaky_features']:.1f}x "
      f"more accurate than it actually is.")

fig, ax = plt.subplots(figsize=(8.6, 4))
labs = [f"{k}  {n}".replace(" + ", "\n+ ").replace(", ", ",\n") for k, n, _ in designs]
vals = [v for _, _, v in designs]
bars = ax.bar(labs, vals, color=["#A11B2A", "#C2601B", "#D9A441", "#1E7A4D"], width=.62)
for r, v in zip(bars, vals):
    ax.text(r.get_x() + r.get_width() / 2, v + .06, f"{v:.2f}%",
            ha="center", fontweight="bold")
ax.set_ylabel("day-ahead MAPE (%)"); ax.set_ylim(0, max(vals) * 1.25)
ax.set_title("One model, one dataset, one target — four evaluation designs")
plt.tight_layout()

In `LEAKS.md`, answer with numbers from the table above:

- how much of the inflation came from the leaky **features** (A -> B)?
- how much came from the **split** (B -> C)?
- what does the spread of D tell you that the single number C hides?
- how many times better than the truth was the number your colleague would have reported?

### Mini-quiz

Answer these in your pair before you look at the next cell.

1. Why is `train_test_split(..., shuffle=True)` fatal on a time series?
2. What does the `.shift(1)` before `.rolling(...)` prevent?
3. Recursive or direct — which one compounds its own errors?
4. Why can a boosted tree never forecast a record high?
5. What tells you which lags to include?

### Task 6 — `LEAKS.md` and commit

`LEAKS.md` has two sections by now:

1. **The two leaks in the inherited starter**, each with the line that causes it and the
   mechanism in one sentence.
2. **The four-design table** above, with the sentence a reviewer needs:
   *the number that would have been reported to the control room was 2.3x better than the
   truth, and nothing in the code looked wrong.*

```
git add -A && git commit -m "feat(features): leakage-safe design matrix + direct LightGBM day-ahead forecaster"
```